# Exercise 4: Constrained FST reranking

*Neural Morphology · version 0.1.0*

> Implement the hybrid FST-constrained neural approach described in the chapter's section on neural analysis. For a Turkish sentence, retrieve the Apertium FST candidates for each word (with `analyser.lookup(word)` on the transducer that TurkicNLP downloads). For each candidate, embed the tag string as a character sequence and concatenate with the word's BERT contextual embedding. Train a linear classifier to select among candidates. First report candidate recall (how often the gold analysis is among the candidates), then compare full-bundle accuracy against (a) always taking the FST's first analysis, and (b) the unconstrained BERT tagger from the Hands-On section.

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" "transformers==5.3.0" sentencepiece protobuf scikit-learn stanza requests` (the compiled `hfst` package needs Python 3.11 or 3.12). Tokenisers are loaded from Hugging Face, or from a local copy in TurkicNLP's model folder (`~/.turkicnlp/models/huggingface/<org>--<name>`). Universal Dependencies files are read from the UD GitHub repositories (master branch), so internet access is needed. Cells whose model cannot be loaded print the reason instead of a result.

Two simplifications make this runnable without a GPU or a downloaded BERT model, and both are part of the answer:

- **A shared label space.** The Apertium readings are converted to UD with TurkicNLP's tag map, but IMST annotates features that the map does not produce (for example `Aspect=Perf`, `Mood=Ind` and `Polarity=Pos` on many verbs). Gold and candidates are therefore compared on UPOS plus a *core* set of features (case, number, possessor, person, tense, verb form, voice, and negative polarity). Without this step, candidate recall measures the mismatch between conventions rather than the transducer.
- **A sparse reranker instead of BERT.** The linear classifier scores each (word, candidate) pair with features that combine candidate tags with the neighbouring words' endings and the word's own ending. Replacing these features with the word's BERT embedding and a character encoding of the tag string is the exercise as stated (Your turn, 1).

The unconstrained tagger of the Hands-On section is not trained in the book (the chapter reports no fine-tuned accuracy). As a neural reference, the last cell scores the Stanza IMST tagger, which is trained on the same treebank, on the same test words and the same core bundle.

In [ ]:
import os, re, collections, statistics
from pathlib import Path
import requests

UD = "https://raw.githubusercontent.com/UniversalDependencies/{repo}/master/{file}"
WORD = re.compile(r"[^\W_]+")                         # the book's word definition (Scripts and Encoding)

def ud_sentences(repo, file):
    """CoNLL-U sentences as lists of (form, lemma, upos, feats) for integer-ID words, with the sent_id."""
    text = requests.get(UD.format(repo=repo, file=file), timeout=120)
    text.raise_for_status()
    sents, words, sid = [], [], None
    for line in text.text.splitlines():
        if line.startswith("# sent_id"):
            sid = line.split("=", 1)[1].strip()
        elif not line.strip():
            if words:
                sents.append((sid, words))
            words = []
        elif not line.startswith("#"):
            c = line.split("\t")
            if c[0].isdigit():
                words.append((c[1], c[2], c[3], c[5]))
    if words:
        sents.append((sid, words))
    return sents

def load_tokenizer(name):
    """A Hugging Face tokenizer, from TurkicNLP's local model folder if present, otherwise from the Hub."""
    from transformers import AutoTokenizer
    local = Path("~/.turkicnlp/models/huggingface").expanduser() / name.replace("/", "--")
    return AutoTokenizer.from_pretrained(str(local) if (local / "tokenizer_config.json").exists() else name)

def pieces(tok, word):
    """Subword pieces of one word without special tokens and without the word-initial marker."""
    out = []
    for p in tok.tokenize(word):
        p = p.replace("▁", "").replace("Ġ", "")
        out.append(p[2:] if p.startswith("##") else p)
    return [p for p in out if p]

CORE_FEATS = ("Case", "Number", "Number[psor]", "Person", "Person[psor]", "Tense", "VerbForm", "Voice")
DEFAULTS = {("Case", "Nom"), ("Number", "Sing"), ("Person", "3")}

def core(upos, feats):
    """UPOS plus the core features, without the unmarked defaults Case=Nom, Number=Sing and Person=3,
    and with Polarity only if negative. Treebanks differ in whether they write the defaults out."""
    kv = dict(f.split("=", 1) for f in (feats or "").split("|") if "=" in f)
    keep = {k: v for k, v in kv.items() if k in CORE_FEATS and (k, v) not in DEFAULTS}
    if kv.get("Polarity") == "Neg":
        keep["Polarity"] = "Neg"
    return upos + "|" + "|".join(f"{k}={v}" for k, v in sorted(keep.items()))

In [ ]:
import sys
from turkicnlp import Pipeline

tur = Pipeline("tur", processors=["tokenize", "morph"], tokenize_backend="rule", morph_backend="apertium")
ap = next(p for p in tur._processors if hasattr(p, "_analyze"))

def candidates(word):
    """Distinct core bundles of the transducer's readings without '+' continuations, in the transducer's order."""
    out = []
    for r in ap._analyze_with_fallback(word):                    # private API in 0.5.0
        if "+" in r.get("raw", ""):
            continue
        upos = ap._tag_mapper.to_ud_pos(r["pos"])
        feats = ap._normalize_ud_feats_for_upos(upos, ap._tag_mapper.to_ud_feats(list(r["feats"]), pos=r["pos"]))
        c = core(upos, feats)
        if c not in out:
            out.append(c)
    return out

train = ud_sentences("UD_Turkish-IMST", "tr_imst-ud-train.conllu")
test = ud_sentences("UD_Turkish-IMST", "tr_imst-ud-test.conllu")
def items(sents):
    rows = []
    for _, ws in sents:
        for i, (form, lemma, upos, feats) in enumerate(ws):
            if upos == "PUNCT":
                continue
            prev_w = ws[i - 1][0].lower() if i else "<s>"
            next_w = ws[i + 1][0].lower() if i + 1 < len(ws) else "</s>"
            rows.append((form, prev_w, next_w, core(upos, feats), candidates(form)))
    return rows
train_items, test_items = items(train), items(test)
for name, rows in [("train", train_items), ("test", test_items)]:
    known = [r for r in rows if r[4]]
    recall = sum(r[3] in r[4] for r in known) / len(known)
    print(f"{name}: {len(rows)} words, {len(known)} with candidates; candidate recall {recall:.3f}; "
          f"mean candidates {statistics.mean(len(r[4]) for r in known):.2f}")

In [ ]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

def pair_features(form, prev_w, next_w, cand):
    f = {"cand=" + cand: 1}
    for part in cand.split("|"):
        f["tag=" + part] = 1
        f["tag=" + part + "&next2=" + next_w[-2:]] = 1
        f["tag=" + part + "&prev2=" + prev_w[-2:]] = 1
        f["tag=" + part + "&final=" + str(next_w in ("</s>", ".", "?", "!"))] = 1
        f["tag=" + part + "&end3=" + form.lower()[-3:]] = 1
    return f

X, y = [], []
for form, p, n, gold, cands in train_items:
    if gold in cands and len(cands) > 1:
        for c in cands:
            X.append(pair_features(form, p, n, c)); y.append(int(c == gold))
vec = DictVectorizer()
clf = LogisticRegression(max_iter=3000, C=0.5).fit(vec.fit_transform(X), y)

def rerank(form, p, n, cands):
    if len(cands) == 1:
        return cands[0]
    scores = clf.predict_proba(vec.transform([pair_features(form, p, n, c) for c in cands]))[:, 1]
    return cands[int(scores.argmax())]

known = [r for r in test_items if r[4]]
first = sum(r[4][0] == r[3] for r in known) / len(known)
learned = sum(rerank(r[0], r[1], r[2], r[4]) == r[3] for r in known) / len(known)
oracle = sum(r[3] in r[4] for r in known) / len(known)
print(f"test words with candidates: {len(known)}")
print(f"first candidate {first:.3f} | learned reranker {learned:.3f} | candidate recall (upper bound) {oracle:.3f}")

In [ ]:
try:
    import stanza
    from turkicnlp.resources.registry import ModelRegistry
    nlp = stanza.Pipeline("tr", processors="tokenize,pos", tokenize_pretokenized=True, verbose=False,
                          model_dir=str(ModelRegistry.default_dir() / "stanza"), download_method=None)
    gold_rows = [(form, core(upos, feats)) for _, ws in test for form, _, upos, feats in ws]
    doc = nlp([[form for form, *_ in ws] for _, ws in test])
    pred = [core(w.upos, w.feats) for s in doc.sentences for w in s.words]
    keep = [(g, p) for (form, g), p in zip(gold_rows, pred) if not g.startswith("PUNCT")]
    print(f"Stanza IMST tagger (trained on IMST train), core-bundle accuracy on all {len(keep)} non-punctuation test words: "
          f"{sum(g == p for g, p in keep) / len(keep):.3f}")
except Exception as exc:
    print("Stanza not available here:", type(exc).__name__, str(exc)[:120])

## Worked solution

Output of `experiments/ch07_exercises/run_ch07_exercises.py` on 19 October 2026 (UD Turkish-IMST, GitHub master branch; TurkicNLP 0.5.0 Apertium transducer; Stanza 1.15.0):

```
train: 31009 words, 26684 with candidates; candidate recall 0.736; mean candidates 2.05
test: 8099 words, 6990 with candidates; candidate recall 0.728; mean candidates 2.03
```

```
test words with candidates: 6990
first candidate 0.462 | learned reranker 0.670 | candidate recall (upper bound) 0.728
```

```
Stanza IMST tagger (trained on IMST train), core-bundle accuracy on all 8099 non-punctuation test words: 0.890
```

**Candidate recall first.** After the convention step (UPOS plus core features, without the unmarked defaults that IMST writes out and the TurkicNLP tag map does not), the transducer has candidates for 6,990 of the 8,099 non-punctuation test words (86%), and the gold core bundle is among the candidates for 72.8% of those. This is the ceiling for any reranker. Without the default-removal step, the same check gave 35.4%: most of the apparent misses were IMST's `Number=Sing|Person=3` on nouns, which the tag map never produces. The remaining misses mix transducer gaps with real convention differences, such as participle against verbal noun for *-DIK* forms (`VerbForm=Part` against `Vnoun`) and UPOS for words like *Evet* 'yes' (NOUN in IMST, INTJ in Apertium).

**Then selection.** Taking the transducer's first candidate is right for 46.2% of the words with candidates; the learned reranker reaches 67.0%, which closes most of the distance to the 72.8% ceiling. Words without candidates count as errors for the whole FST pipeline, which therefore gets 57.8% of all 8,099 words right (0.670 × 6,990 / 8,099).

**Against a neural tagger.** The Stanza IMST tagger, trained on the same treebank, gets 89.0% of all 8,099 words right on the same core bundle. It is not limited by the transducer's candidates, and it learns IMST's conventions directly. The reranker would need a much better candidate set, or contextual features such as the BERT embeddings of the exercise, to compete; the constrained approach pays off where no treebank exists to train such a tagger, not where one does.

## Your turn — extension

1. Replace `pair_features` with the concatenation of the word's BERTurk embedding (first subword) and a character n-gram encoding of the candidate string, as the exercise asks, and compare with the sparse reranker on the same split.
2. List the twenty most frequent test words whose gold core bundle is not among the candidates. How many are transducer gaps and how many are convention differences between Apertium and IMST?